# 🩺 RAG Fairness Audit — Qwen2.5 3B + Fairlearn

This notebook audits whether a local RAG pipeline built on **Qwen2.5-3B-Instruct** produces
equally good answers across demographic groups (gender × age).

### Pipeline
```
Medical Documents
      ↓ SentenceTransformers (all-MiniLM-L6-v2)
  FAISS Index
      ↓ Top-3 retrieval
  Qwen2.5-3B-Instruct  →  Answer
      ↓
  Fairlearn MetricFrame  →  Fairness Report
```

### Sensitive Features Audited
- **Gender**: `female` vs `male`
- **Age group**: `older` vs `younger`

---

## 0. Install Dependencies

In [ ]:
# Run once — skip if already installed
!pip install -q faiss-cpu sentence-transformers fairlearn transformers torch datasets matplotlib

## 1. Imports & Project Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path so we can import our modules
sys.path.insert(0, './src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from dataset import MEDICAL_DOCUMENTS, build_fairness_dataset
from rag_pipeline import FAISSDocumentStore, QwenGenerator, RAGPipeline
from fairness_eval import RAGFairnessEvaluator
from visualize import full_dashboard

print('✅ All imports successful')

## 2. Load Documents into FAISS

In [ ]:
# Preview the corpus
print(f'Total documents: {len(MEDICAL_DOCUMENTS)}\n')
for i, doc in enumerate(MEDICAL_DOCUMENTS[:3]):
    print(f'[{i}] {doc[:120]}...')
print('...')

In [ ]:
# Build the FAISS document store
store = FAISSDocumentStore(embed_model='all-MiniLM-L6-v2')
store.add_documents(MEDICAL_DOCUMENTS)
print('\n✅ FAISS index ready')

In [ ]:
# Quick retrieval sanity check
test_query = 'What are the risks of heart disease for older women?'
results = store.retrieve(test_query, top_k=3)
print(f'Query: "{test_query}"\n')
for i, (doc, dist) in enumerate(results):
    print(f'[{i+1}] dist={dist:.3f}  {doc[:100]}...')

## 3. Load Qwen2.5-3B-Instruct

> ⚠️ Requires ~6 GB VRAM (GPU) or ~12 GB RAM (CPU).  
> Set `device_map='cpu'` in `rag_pipeline.py` if needed.

In [ ]:
generator = QwenGenerator(model_name='Qwen/Qwen2.5-3B-Instruct')
print('\n✅ Qwen2.5-3B ready')

## 4. Assemble the RAG Pipeline

In [ ]:
pipeline = RAGPipeline(document_store=store, generator=generator, top_k=3)
print('✅ Pipeline assembled')

In [ ]:
# End-to-end test
result = pipeline.run('What are the risks of heart disease for older women?')
print('Question:', result['query'])
print('\nAnswer:\n', result['answer'][:500])

## 5. Fairness Dataset

16 medical queries — same topic, different gender × age combinations.

In [ ]:
dataset = build_fairness_dataset()
print(f'Dataset shape: {dataset.shape}')
dataset[['topic', 'query', 'gender', 'age_group']].head(8)

In [ ]:
print('Distribution:')
print(dataset.groupby(['gender', 'age_group']).size().to_string())

## 6. Run Evaluation

In [ ]:
evaluator = RAGFairnessEvaluator(pipeline=pipeline)
results = evaluator.evaluate(dataset)
print('\n✅ Evaluation complete')

In [ ]:
# Preview answers and scores
display_cols = ['topic', 'gender', 'age_group', 'keyword_score', 'refusal_score', 'composite_label']
results[display_cols].style.background_gradient(
    subset=['keyword_score', 'refusal_score'], cmap='RdYlGn', vmin=0, vmax=1
)

## 7. Fairness Report — Gender

In [ ]:
evaluator.print_report('gender')

## 8. Fairness Report — Age Group

In [ ]:
evaluator.print_report('age_group')

## 9. Visualise — Full Fairness Dashboard

In [ ]:
report_gender = evaluator.compute_fairness_report('gender')
report_age    = evaluator.compute_fairness_report('age_group')

fig = full_dashboard(results, report_gender, report_age)
fig.savefig('./outputs/fairness_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved → outputs/fairness_dashboard.png')

## 10. Drill-Down — Which Topics Are Least Fair?

In [ ]:
topic_gender = results.groupby(['topic', 'gender'])['keyword_score'].mean().unstack()
topic_gender['disparity'] = (topic_gender['female'] - topic_gender['male']).abs()
topic_gender = topic_gender.sort_values('disparity', ascending=False)

print('Gender Disparity by Topic (keyword score gap):')
print(topic_gender.to_string())

# Plot
ax = topic_gender[['female', 'male']].plot(
    kind='bar', figsize=(10, 5),
    color=['#E07B91', '#5B9BD5'], edgecolor='white'
)
ax.set_title('Keyword Score by Topic and Gender', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Keyword Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.axhline(0.5, linestyle='--', color='red', linewidth=1, label='Threshold')
ax.legend()
plt.tight_layout()
plt.savefig('./outputs/topic_gender_disparity.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Inspect Raw Answers for Biased Topics

In [ ]:
# Change this to any topic that showed high disparity
TOPIC = 'mental_health'

subset = results[results['topic'] == TOPIC][['gender', 'age_group', 'query', 'answer', 'keyword_score']]
for _, row in subset.iterrows():
    print(f'\n--- {row["gender"]} / {row["age_group"]} | score={row["keyword_score"]:.2f} ---')
    print(f'Q: {row["query"]}')
    print(f'A: {row["answer"][:300]}...')

## 12. Mitigation Strategy

Since we cannot retrain Qwen, we use two post-hoc strategies:

### Strategy A — Prompt Engineering
Re-run biased queries with an explicitly fair system prompt.

### Strategy B — ThresholdOptimizer (classification layer)
Adjust classification thresholds per group to equalise accuracy.

In [ ]:
# Strategy A: Fair prompt re-run
def generate_fair_answer(query: str, store, generator) -> str:
    """Use an equity-aware prompt."""
    retrieved = store.retrieve(query, top_k=3)
    context = '\n\n'.join(f'[Doc {i+1}]: {doc}' for i, (doc, _) in enumerate(retrieved))
    
    prompt = (
        'You are a fair and unbiased medical assistant. '
        'Provide equally detailed and accurate answers regardless of the '
        'patient\'s gender, age, or background. '
        'Use only the context below.\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {query}\n'
        'Answer:'
    )
    
    import torch
    inputs = generator.tokenizer(prompt, return_tensors='pt').to(generator.model.device)
    with torch.no_grad():
        output_ids = generator.model.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            pad_token_id=generator.tokenizer.eos_token_id
        )
    full = generator.tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full.split('Answer:')[-1].strip()

print('Function defined. Run on biased queries to compare scores.')

In [ ]:
# Strategy B: ThresholdOptimizer
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.dummy import DummyClassifier
import numpy as np

# Use keyword_score as a probability proxy (float in [0,1])
X_scores = results[['keyword_score', 'length_score', 'refusal_score']].values
y_true = np.ones(len(results), dtype=int)
y_pred_proba = results['keyword_score'].values
sensitive = results['gender']

# Wrap scores as a simple classifier
class ScoreClassifier:
    def fit(self, X, y): return self
    def predict(self, X): return (X[:, 0] >= 0.5).astype(int)
    def predict_proba(self, X):
        p = X[:, 0]
        return np.column_stack([1-p, p])

base_clf = ScoreClassifier()

threshold_opt = ThresholdOptimizer(
    estimator=base_clf,
    constraints='demographic_parity',
    objective='accuracy_score',
    predict_method='predict_proba'
)

threshold_opt.fit(X_scores, y_true, sensitive_features=sensitive)
y_pred_fair = threshold_opt.predict(X_scores, sensitive_features=sensitive)

# Compare before vs after
from fairlearn.metrics import demographic_parity_difference
before = demographic_parity_difference(y_true, results['composite_label'], sensitive_features=sensitive)
after  = demographic_parity_difference(y_true, y_pred_fair, sensitive_features=sensitive)

print(f'Demographic Parity Difference')
print(f'  Before ThresholdOptimizer : {before:.4f}')
print(f'  After  ThresholdOptimizer : {after:.4f}')
print(f'  Improvement               : {before - after:.4f}')

## 13. Save Results

In [ ]:
import os
os.makedirs('./outputs', exist_ok=True)

# Save full results table
results.drop(columns=['expected_keywords']).to_csv('./outputs/evaluation_results.csv', index=False)

# Save group summaries
results.groupby('gender')[['keyword_score','composite_label']].mean().to_csv('./outputs/gender_summary.csv')
results.groupby('age_group')[['keyword_score','composite_label']].mean().to_csv('./outputs/age_summary.csv')

print('✅ Saved to ./outputs/')
print('   - evaluation_results.csv')
print('   - gender_summary.csv')
print('   - age_summary.csv')
print('   - fairness_dashboard.png')
print('   - topic_gender_disparity.png')

## Summary

| Component | Details |
|---|---|
| **Documents** | 19 medical paragraphs |
| **Vector DB** | FAISS (L2, top-3 retrieval) |
| **Embedder** | `all-MiniLM-L6-v2` |
| **Generator** | `Qwen2.5-3B-Instruct` |
| **Sensitive features** | `gender`, `age_group` |
| **Fairness metrics** | Accuracy, Precision, Recall, DP Difference |
| **Mitigation** | Prompt engineering + ThresholdOptimizer |

### Key Interpretation
- **DP Difference ≤ 0.1** → model is considered fair on that attribute
- **DP Difference > 0.1** → bias detected; inspect topic-level drill-down (Cell 10)
- After mitigation (Cell 12), re-evaluate and compare
